# OpenFEMA Disaster Declarations Raw Data Quality

OpenFEMA Disaster Declarations Summaries v2.

The audit queries the current `raw.*` ingestion tables directly; transformed tables are out of scope.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "data" / "housing_predict.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
con = duckdb.connect(str(ROOT / "data" / "housing_predict.duckdb"), read_only=True)
TABLES = ["fema"]
KEYS = {"fema":["id"]}
YEAR_COLUMNS = {}
EXPECTED_YEARS = set(range(2010, 2025))
def qi(x): return '"' + x.replace('"', '""') + '"'
def columns(t):
    return con.execute("SELECT column_name FROM information_schema.columns WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [t]).df()["column_name"].tolist()
available = set(con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='raw'").df()["table_name"])
assert set(TABLES) <= available, f"Missing raw tables: {set(TABLES)-available}"


## Inventory, coverage, keys, and sampled missingness

In [2]:
inventory, key_checks, coverage, missing = [], [], [], []
for table in TABLES:
    cols = columns(table)
    n = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    inventory.append({"table": table, "rows": n, "columns": len(cols)})
    key = KEYS[table]
    key_sql = ", ".join(qi(c) for c in key)
    invalid = " OR ".join(f"{qi(c)} IS NULL OR trim(cast({qi(c)} AS VARCHAR))=''" for c in key)
    key_checks.append({
        "table": table, "candidate_key": " + ".join(key),
        "invalid_key_rows": con.execute(f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}").fetchone()[0],
        "duplicate_key_groups": con.execute(f"SELECT count(*) FROM (SELECT {key_sql}, count(*) n FROM raw.{qi(table)} GROUP BY {key_sql} HAVING n>1)").fetchone()[0],
    })
    year_col = YEAR_COLUMNS.get(table)
    if year_col:
        years = set(con.execute(f"SELECT DISTINCT try_cast({qi(year_col)} AS INTEGER) FROM raw.{qi(table)} WHERE try_cast({qi(year_col)} AS INTEGER) IS NOT NULL").df().iloc[:,0])
        coverage.append({"table":table, "min_year":min(years), "max_year":max(years), "missing_years":sorted(EXPECTED_YEARS-years), "outside_scope":sorted(years-EXPECTED_YEARS)})
    # Identifier fields plus a bounded sample of wide value fields keeps the audit runnable.
    profile_cols = list(dict.fromkeys([*key, *(["NAME"] if "NAME" in cols else []), *cols[:40]]))
    for col in profile_cols:
        m = con.execute(f"SELECT count(*) FROM raw.{qi(table)} WHERE {qi(col)} IS NULL OR trim(cast({qi(col)} AS VARCHAR))=''").fetchone()[0]
        if m: missing.append({"table":table,"column":col,"missing_rows":m,"missing_pct":round(100*m/n,2)})
display(pd.DataFrame(inventory))
display(pd.DataFrame(coverage))
display(pd.DataFrame(key_checks))
display(pd.DataFrame(missing).sort_values("missing_pct",ascending=False).head(100) if missing else pd.DataFrame())


,table,rows,columns
0,fema,21983,28


""


,table,candidate_key,invalid_key_rows,duplicate_key_groups
0,fema,id,0,2


,table,column,missing_rows,missing_pct
3,fema,designatedIncidentTypes,17859,81.24
2,fema,lastIAFilingDate,15297,69.59
1,fema,disasterCloseoutDate,9788,44.53
0,fema,incidentEndDate,159,0.72


## Source-specific checks

In [3]:
excluded=["Biological","Dam/Levee Break","Chemical","Terrorist","Other","Toxic Substances"]
display(con.execute("""
SELECT count(*) FILTER (WHERE (try_cast(incidentBeginDate AS TIMESTAMPTZ) BETWEEN TIMESTAMPTZ '2010-01-01' AND TIMESTAMPTZ '2024-12-31 23:59:59.999' OR try_cast(incidentEndDate AS TIMESTAMPTZ) BETWEEN TIMESTAMPTZ '2010-01-01' AND TIMESTAMPTZ '2024-12-31 23:59:59.999')) AS scoped_rows,
 count(*) FILTER (WHERE incidentType IN (SELECT * FROM unnest(?))) AS excluded_incident_rows,
 count(*) FILTER (WHERE lower(trim(designatedArea)) IN ('statewide','state') OR trim(fipsCountyCode) IN ('000','')) AS statewide_rows,
 count(DISTINCT disasterNumber) AS disasters
FROM raw.fema
""",[excluded]).df())
display(con.execute("""SELECT incidentType, count(*) AS row_count, count(DISTINCT disasterNumber) AS disasters FROM raw.fema WHERE try_cast(fyDeclared AS INTEGER) BETWEEN 2010 AND 2024 GROUP BY 1 ORDER BY row_count DESC""").df())

,scoped_rows,excluded_incident_rows,statewide_rows,disasters
0,21983,0,350,1820


,incidentType,row_count,disasters
0,Severe Storm,6853,438
1,Hurricane,6542,211
2,Flood,3384,218
3,Severe Ice Storm,1363,27
4,Fire,1264,738
5,Tropical Storm,926,28
6,Snowstorm,516,31
7,Tornado,300,30
8,Coastal Storm,184,9
9,Earthquake,137,11


## Interpretation and limitations

The raw table is filtered at the API to incidents whose begin or end date falls from January 1, 2010 through December 31, 2024, and excludes Biological, Dam/Levee Break, Chemical, Terrorist, Other, and Toxic Substances. A disaster has multiple designated-area rows; statewide rows require downstream expansion against the county reference.

Displayed failures are investigation items. This notebook does not alter source data.

In [4]:
con.close()